# A Genetikai Kódfejtő alkalmazás logikája

## Általános működés

Az alkalmazás egy **weboldal alapú bioinformatikai eszköz**, amely DNS szekvenciákat elemez és azokat fehérjékké fordítja le. A rendszer három fő komponensből áll: az **HTML felület**, a **CSS stílus** és a **JavaScript logika**.

## 1. Adatkezelés és minták (SAMPLES objektum)

- Az alkalmazás előre betöltött DNS minta-szekvenciákat tartalmaz (`minta_01` - `minta_04`)
- Minden mintához tartozik egy név, leírás és egy valódi (vagy képzeletbeli) DNS szekvencia
- Ez a megközelítés lehetővé teszi a felhasználó számára, hogy gyorsan tesztelhessen különböző szekvenciákat anélkül, hogy saját adatokat kellene beírnia

## 2. Faj kiválasztás és automatikus betöltés

```
Felhasználó kiválaszt egy fajt → onchange esemény
→ loadPreset() függvény meghívódik
→ A választott DNS szekvencia betöltődik a textareaba
→ Faj leírása megjelenik az oldal felhasználó számára
```

Ez a **reaktív tervezési minta** alkalmazása - az adatok azonnal frissülnek az interakció után.

## 3. DNS szanitizálás (sanitizeDna)

```javascript
sanitizeDna(seq) → csak ATGC karaktereket tartja meg, felső case-re alakít
```

- Ez kritikus, mert a DNS szekvenciákat csak ezek az alapok képezik
- Eltávolít szóközöket, sortöréseket és egyéb karaktereket
- **Biztonságot** és **adatintegritást** biztosít

## 4. GC-tartalom kiszámítása (gcContent)

```
DNS szekvencia → G és C bázisok száma
→ (G+C bázisok száma / össz hossz) × 100%
```

- A GC-tartalom a **DNS stabilitásának** mutatója
- Magas GC% = stabilabb, alacsony GC% = kevésbé stabil

In [ ]:
# GC-tartalom kiszámítása - Gyakorlati példa

def gc_content(dna_seq):
    """Kiszámítja a GC-tartalom százalékét"""
    if not dna_seq:
        return 0
    g_count = dna_seq.count('G')
    c_count = dna_seq.count('C')
    gc_percent = ((g_count + c_count) / len(dna_seq)) * 100
    return round(gc_percent, 2)

# Próba
minta_dns = "GCATCGTAGCTAGCCGATGCGCTACGGCTGGCAGTTTGACGTGAACAAATAACGATCGATCGACT"
print(f"DNS szekvencia: {minta_dns}")
print(f"GC-tartalom: {gc_content(minta_dns)}%")

## 5. DNS → mRNS átírás (dnaToMrna)

```
DNS: ...ATGC...
↓ (T → U csere)
mRNS: ...AUGC...
```

- Ez a **transzkrició** első lépése
- A timín (T) ribonukleózidra (U) cserélődik az mRNS-ben

In [ ]:
# DNS → mRNS konverzió

def dna_to_mrna(dna_seq):
    """Átalakítja a DNS szekvenciát mRNS-é (T → U)"""
    return dna_seq.replace('T', 'U')

# Próba
dns = "ATGAAATTT"
mrna = dna_to_mrna(dns)
print(f"DNS:  {dns}")
print(f"mRNS: {mrna}")

## 6. Protein fordítás (translate)

Ez az alkalmazás **legbonyolultabb része**:

```
mRNS szekvencia
↓ (keresi az első AUG start kodont)
Triplett bázisok (3 bázisonként) olvasása
↓
Kodon tábla alapján aminosavra konvertálás
↓ (amíg STOP kodon nem érkezik: UAA, UAG, UGA)
Aminosav sorrend
```

**Fontos koncepciók:**
- **Start kodon (AUG)**: Ahol kezdődik a fehérje szintézis
- **Stop kodon (UAA, UAG, UGA)**: Ahol megáll a fordítás
- **Kodon tábla**: 64 lehetséges triplett → 20 aminosav
- **Leolvasási keret**: Az olvasás az AUG-tól 3-3 bázisonként történik

In [ ]:
# Kodon tábla - Részlet
CODON_TABLE = {
    'AUG': 'Met',  # Start
    'UUU': 'Phe', 'UUC': 'Phe',
    'UUA': 'Leu', 'UUG': 'Leu',
    'CUU': 'Leu', 'CUC': 'Leu', 'CUA': 'Leu', 'CUG': 'Leu',
    'UAA': 'STOP', 'UAG': 'STOP', 'UGA': 'STOP',
    'AUA': 'Ile', 'AUU': 'Ile', 'AUC': 'Ile',
    'GCU': 'Ala', 'GCC': 'Ala', 'GCA': 'Ala', 'GCG': 'Ala',
    'GAA': 'Glu', 'GAG': 'Glu',
    'AAA': 'Lys', 'AAG': 'Lys',
}

print("Kodon tábla minta:")
for codon, aa in list(CODON_TABLE.items())[:10]:
    print(f"{codon} → {aa}")

In [ ]:
# Protein fordítás függvény

def translate(mrna_seq, codon_table):
    """Fordítja az mRNS szekvenciát aminosavsorrendé"""
    
    # Keresi az első AUG start kodont
    start = mrna_seq.find('AUG')
    if start == -1:
        return [], None  # Nincs start kodon
    
    amino_acids = []
    pos = start
    
    # Olvassa 3-3 bázisonként (kodont)
    while pos + 3 <= len(mrna_seq):
        codon = mrna_seq[pos:pos+3]
        aa_name = codon_table.get(codon, '???')
        
        if aa_name == 'STOP':
            return amino_acids, (start, pos + 3)  # Stop kodon találtam
        
        amino_acids.append(aa_name)
        pos += 3
    
    return amino_acids, (start, pos)

# Próba
mrna_test = "AUGAAAUUUUAA"  # Met-Lys-Phe-STOP
amino_acids, region = translate(mrna_test, CODON_TABLE)
print(f"mRNS: {mrna_test}")
print(f"Aminosavak: {amino_acids}")
print(f"START/STOP régió: {region}")

## 7. Teljes analízis folyamat

```
analyze() függvény:
  ↓
  DNS szanitizálása
  ↓
  GC-tartalom számítás
  ↓
  mRNS konverzió + START/STOP terület kiemelése
  ↓
  Protein fordítás
  ↓
  Eredmények HTML-ben megjelenítése
  ↓
  Automatikus scroll az output szekcióhoz
```

In [ ]:
# Teljes elemzés folyamat szimulációja

def sanitize_dna(seq):
    """Megtisztítja a DNS szekvenciát"""
    return ''.join([c for c in seq.upper() if c in 'ATGC'])

def analyze_dna(dna_input, codon_table):
    """Teljes DNS analízis"""
    
    # 1. Szanitizálás
    dna = sanitize_dna(dna_input)
    print(f"1. Megtisztított DNS: {dna}")
    
    if not dna:
        print("Hiba: Nincs érvényes DNS!")
        return
    
    # 2. GC-tartalom
    gc = gc_content(dna)
    print(f"2. GC-tartalom: {gc}%")
    
    # 3. mRNS konverzió
    mrna = dna_to_mrna(dna)
    print(f"3. mRNS: {mrna}")
    
    # 4. Protein fordítás
    amino_acids, region = translate(mrna, codon_table)
    print(f"4. Aminosavak: {amino_acids}")
    if region:
        print(f"   START/STOP régió: {region[0]}-{region[1]}")

# Próba valós mintával
print("=== TELJES ANALÍZIS ===")
test_dna = "GCATCGTAGCTAGCCGATGCGCTACGGCTGGCAGTTTGACGTGAACAAATAACGATCGATCGACT"
analyze_dna(test_dna, CODON_TABLE)

## Architekturális megközelítés

- **Deklaratív felhasználói felület**: Az HTML strukturált, könnyű értelmezéshez
- **Szeparáció**: Logika (JS), megjelenítés (CSS), tartalom (HTML)
- **Valós idejű visszajelzés**: Az "onchange" esemény azonnal frissíti az adatokat
- **Felhasználó-barát**: Clear gombal bármikor nullázható az összes adat

## Összefoglalás

Az alkalmazás **biológiai folyamatok digitális szimulációja**, amely:

1. **Adatot** fogad (DNS)
2. **Transzformálja** azt (transzkrició, fordítás)
3. **Megjelenít** eredményeket (GC%, mRNS, aminosavak)

Ez egy **edukatív eszköz**, amely segít megérteni a genetikai kódfejtés alapvető mechanizmusát.